# Homework 04: Data Acquisition and Ingestion

One API pull (yfinance, since no real Alpha Vantage key is configured) and one scraped table (Wikipedia's S&P 500 constituent list), each validated and saved as a timestamped CSV in `data/raw/`.

In [1]:
import os, pathlib, datetime as dt
import requests
import pandas as pd
from bs4 import BeautifulSoup
from dotenv import load_dotenv

RAW = pathlib.Path('data/raw'); RAW.mkdir(parents=True, exist_ok=True)
load_dotenv()
print('ALPHAVANTAGE_API_KEY loaded?', bool(os.getenv('ALPHAVANTAGE_API_KEY')))

ALPHAVANTAGE_API_KEY loaded? False


## Helpers

In [2]:
def ts():
    return dt.datetime.now().strftime('%Y%m%d-%H%M%S')

def save_csv(df: pd.DataFrame, prefix: str, **meta):
    mid = '_'.join([f"{k}-{v}" for k, v in meta.items()])
    path = RAW / f"{prefix}_{mid}_{ts()}.csv"
    df.to_csv(path, index=False)
    print('Saved', path)
    return path

def validate(df: pd.DataFrame, required):
    missing = [c for c in required if c not in df.columns]
    return {'missing': missing, 'shape': df.shape, 'na_total': int(df.isna().sum().sum())}

## Part 1: API Pull (required)

`ALPHAVANTAGE_API_KEY` isn't set. Free-tier keys are personal, so a dummy value won't work against a real endpoint, and this falls back to `yfinance`, exactly as the assignment allows ("requests, or yfinance as fallback"). Ticker: `AAPL`, daily bars, 3 months.

In [3]:
SYMBOL = 'AAPL'
USE_ALPHA = bool(os.getenv('ALPHAVANTAGE_API_KEY'))

if USE_ALPHA:
    url = 'https://www.alphavantage.co/query'
    params = {'function': 'TIME_SERIES_DAILY', 'symbol': SYMBOL, 'outputsize': 'compact',
              'apikey': os.getenv('ALPHAVANTAGE_API_KEY')}
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    js = r.json()
    key = [k for k in js if 'Time Series' in k]
    if not key:
        print('Alpha Vantage returned no series:', str(list(js.values())[0])[:150])
        USE_ALPHA = False

if USE_ALPHA:
    df_api = pd.DataFrame(js[key[0]]).T.reset_index().rename(columns={'index': 'date', '4. close': 'close'})[['date', 'close']]
    df_api['date'] = pd.to_datetime(df_api['date'])
    df_api['close'] = pd.to_numeric(df_api['close'])
else:
    import yfinance as yf
    df_api = yf.download(SYMBOL, period='3mo', interval='1d', auto_adjust=False,
                          multi_level_index=False, progress=False).reset_index()[['Date', 'Close']]
    df_api.columns = ['date', 'close']

v_api = validate(df_api, ['date', 'close'])
print(v_api)
df_api.head()

{'missing': [], 'shape': (65, 2), 'na_total': 0}


,date,close
0,2026-05-27,310.850006
1,2026-05-28,312.510010
2,2026-05-29,312.059998
3,2026-06-01,306.309998
4,2026-06-02,315.200012


In [4]:
api_path = save_csv(df_api.sort_values('date'), prefix='api', source='alpha' if USE_ALPHA else 'yfinance', symbol=SYMBOL)

Saved data\raw\api_source-yfinance_symbol-AAPL_20260827-135603.csv


## Part 2: Scrape a Public Table (required)

Source: [Wikipedia, List of S&P 500 companies](https://en.wikipedia.org/wiki/List_of_S%26P_500_companies), the `id="constituents"` table. Wikipedia article pages are scrape-permitted (see `en.wikipedia.org/robots.txt`), the table markup is stable, and it's directly market-related.

In [5]:
SCRAPE_URL = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
headers = {'User-Agent': 'FRE5040-Homework/1.0'}

resp = requests.get(SCRAPE_URL, headers=headers, timeout=30)
resp.raise_for_status()
soup = BeautifulSoup(resp.text, 'html.parser')

table = soup.find('table', {'id': 'constituents'})
rows = [[c.get_text(strip=True) for c in tr.find_all(['th', 'td'])] for tr in table.find_all('tr')]
header, *data = [r for r in rows if r]
df_scrape = pd.DataFrame(data, columns=header)
df_scrape = df_scrape[['Symbol', 'Security', 'GICSSector', 'GICS Sub-Industry']]

v_scrape = validate(df_scrape, ['Symbol', 'Security', 'GICSSector', 'GICS Sub-Industry'])
print(v_scrape)
df_scrape.head()

{'missing': [], 'shape': (503, 4), 'na_total': 0}


,Symbol,Security,GICSSector,GICS Sub-Industry
0,MMM,3M,Industrials,Industrial Conglomerates
1,AOS,A. O. Smith,Industrials,Building Products
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment
3,ABBV,AbbVie,Health Care,Biotechnology
4,ACN,Accenture,Information Technology,IT Consulting & Other Services


In [6]:
scrape_path = save_csv(df_scrape, prefix='scrape', site='wikipedia', table='sp500-constituents')

Saved data\raw\scrape_site-wikipedia_table-sp500-constituents_20260827-135604.csv


## Documentation

**API source:** `yfinance` (Yahoo Finance), ticker `AAPL`, `period=3mo`, `interval=1d`, `auto_adjust=False`. No key is required for this path. `ALPHAVANTAGE_API_KEY` is read from `.env`, and the notebook would use the real Alpha Vantage `TIME_SERIES_DAILY` endpoint instead if a real key were present.

**Scrape source:** Wikipedia, *List of S&P 500 companies*, the `#constituents` table, columns `Symbol`, `Security`, `GICS Sector`, `GICS Sub-Industry`. Parsed manually with BeautifulSoup (`find('table', {'id': 'constituents'})` then row by row `<tr>`/`<td>` extraction) rather than `pandas.read_html`, per the assignment.

**Validation logic:** `validate()` checks for required columns and counts total NAs and shape for both pulls. Both come back with zero missing required columns and, checked above, no NAs.

**Assumptions and risks:**
- Wikipedia's table markup (the `id="constituents"` selector) could change or get renamed, which would silently break the scraper instead of raising an obvious error. That's a resilience gap worth a schema check in a production version.
- Yahoo Finance's unofficial API (used by `yfinance`) has no SLA and can rate-limit or change shape without notice. Alpha Vantage would be the more stable production source once a real key is available.
- `.env` holds only `ALPHAVANTAGE_API_KEY=` (blank) and `DATA_DIR`. It's listed in the root `.gitignore` and isn't committed. `.env.example`, which is committed, documents the expected keys.